# X1_floor_effect_correct

**Buries:** the floor-effect confound — replaces the contested 'RESOLVED' single-trend with three probes (matched class count / matched accuracy / dataset-clustered pooled model).

In [ ]:
import sys, os
# --- L1 dataset location (edit here if your data ever moves) -------------------------
os.environ.setdefault('SEMG_L1_ROOT', '/home/honors/Ashraf_Ali_2025_Batch/data/L1')
cwd = os.getcwd()
cands = [cwd, os.path.dirname(cwd), os.path.join(cwd, 'notebooks')]
PROFILE = next((p for p in cands if os.path.isdir(os.path.join(p, 'dsprofile'))), cwd)
if PROFILE not in sys.path: sys.path.insert(0, PROFILE)
print('PROFILE =', PROFILE)

In [ ]:
from cli import floor_effect_x1 as fx
assert fx.selftest(), 'X1 GROUND TRUTH FAILED'

In [ ]:
from paper_experiments import common
DATASETS = list(common.config.ALL14)   # all 14 -> more cohort clusters -> tighter CIs (long run)
JOBS = 4
per_dataset, all_rungs = {}, []
outdir = common.results_dir('floor_effect_x1')
for ds in DATASETS:
    try:
        out, rungs = fx.run_dataset(ds, target_classes=17, target_acc=0.15, n_subsets=20, seed=42, n_jobs=JOBS)
    except Exception as e:
        print('[FAIL]', ds, e); per_dataset[ds] = {'error': str(e)}; continue
    common.atomic_write_json(outdir / f'{ds}__x1.json', out)
    per_dataset[ds] = out; all_rungs += rungs; print('[OK]', ds)
pooled = fx.build_pooled(all_rungs, per_dataset, bootstrap=2000, seed=42)
common.atomic_write_json(outdir / 'pooled.json', pooled)
print(pooled.get('verdict', pooled))